# Discrete-Time Riccati Equation
## From Backward Induction to LQG — Discrete Optimal Control

---

**Author:** Computational Mathematics Notebook Series  
**Topic:** DARE, Discrete-Time LQR, Riccati Recursion, LQG  
**Prerequisites:** Linear algebra, control theory basics, discrete-time systems  
**Primary Reference:** Anderson, B. D. O., & Moore, J. B. (1990). *Optimal Control: Linear Quadratic Methods*. Prentice-Hall.

In [ ]:
%matplotlib inline

import numpy as np
from scipy import linalg as la
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2
})

print("All imports successful.")

In [ ]:
# =============================================================================
# Global Constants
# =============================================================================

SEED = 42
np.random.seed(SEED)

# Solver parameters
DARE_TOL      = 1e-12   # Convergence tolerance for iterative DARE
DARE_MAX_ITER = 5000    # Maximum iterations for value iteration
SCHUR_TOL     = 1e-10   # Tolerance for Schur-based solver

# Simulation
N_STEPS       = 50      # Finite-horizon episode length
N_SIM         = 80      # Closed-loop trajectory simulation steps

# Stability threshold
STABILITY_TOL = 1.0     # Discrete-time: eigenvalues must have |lambda| < 1

# =============================================================================
# System matrices used throughout the notebook
# =============================================================================

# System 1: "Old notebook" reference system (2-state, 1-input)
A1 = np.array([[ 0.16,  2.16],
               [-0.16, -1.16]])
B1 = np.array([[-1.0],
               [ 1.0]])
Q1 = np.eye(2)
R1 = np.array([[0.1]])

# System 2: Discretized double integrator  (Ts = 0.1 s)
Ts = 0.1
A2 = np.array([[1.0, Ts],
               [0.0, 1.0]])
B2 = np.array([[0.5 * Ts**2],
               [Ts]])
Q2 = np.diag([10.0, 1.0])
R2 = np.array([[1.0]])

# System 3: 4-state discretized spring-mass-damper chain
# Continuous: x1_dot = x2, x2_dot = -2x1 + x3 - 0.5x2
#             x3_dot = x4, x4_dot = x1 - 2x3 - 0.5x4 + u
Ac = np.array([[ 0.0,  1.0,  0.0,  0.0],
               [-2.0, -0.5,  1.0,  0.0],
               [ 0.0,  0.0,  0.0,  1.0],
               [ 1.0,  0.0, -2.0, -0.5]])
Bc = np.array([[0.0], [0.0], [0.0], [1.0]])
Ts3 = 0.05
# Zero-order hold discretization
n3 = Ac.shape[0]
M = np.zeros((n3 + 1, n3 + 1))
M[:n3, :n3] = Ac * Ts3
M[:n3, n3:] = Bc * Ts3
eM = la.expm(M)
A3 = eM[:n3, :n3]
B3 = eM[:n3, n3:]
Q3 = np.eye(4)
R3 = np.array([[0.5]])

# Plot colors
C_BLUE  = 'steelblue'
C_RED   = 'coral'
C_GREEN = 'seagreen'
C_GOLD  = 'goldenrod'
C_PURP  = 'mediumpurple'

print(f"System 1  A={A1.shape}, B={B1.shape}")
print(f"System 2  A={A2.shape}, B={B2.shape}")
print(f"System 3  A={A3.shape}, B={B3.shape}")
print("Constants loaded.")

---
## 1. Problem Statement — Discrete-Time LQR

### Discrete-Time Linear System

Consider the discrete-time LTI system:

$$x_{k+1} = A x_k + B u_k, \qquad x_0 \text{ given}$$

where $x_k \in \mathbb{R}^n$ is the state, $u_k \in \mathbb{R}^m$ the control input, $A \in \mathbb{R}^{n \times n}$ the state transition matrix, and $B \in \mathbb{R}^{n \times m}$ the input matrix.

### Infinite-Horizon Cost

We minimize the infinite-horizon quadratic cost:

$$\boxed{J = \sum_{k=0}^{\infty} \left( x_k^T Q x_k + u_k^T R u_k \right)}$$

where $Q \geq 0$ (positive semidefinite) penalizes state deviation and $R > 0$ (positive definite) penalizes control effort.

### The Optimal Control Law

Under the assumptions that $(A, B)$ is stabilizable and $(A, Q^{1/2})$ is detectable, the optimal control law is a **linear state feedback**:

$$u_k^* = -K x_k, \qquad K = (R + B^T P B)^{-1} B^T P A$$

where $P$ is the **unique positive semidefinite solution** of the Discrete Algebraic Riccati Equation (DARE):

$$\boxed{P = Q + A^T P A - A^T P B (R + B^T P B)^{-1} B^T P A}$$

### Key Differences from Continuous-Time LQR

| Property | Continuous-time | Discrete-time |
|----------|----------------|---------------|
| Riccati equation | $A^T P + PA - PBR^{-1}B^T P + Q = 0$ | $P = Q + A^T PA - A^T PB(R+B^T PB)^{-1}B^T PA$ |
| Optimal gain | $K = R^{-1}B^T P$ | $K = (R + B^T PB)^{-1}B^T PA$ |
| Stability condition | $\text{Re}(\lambda_i(A-BK)) < 0$ | $|\lambda_i(A-BK)| < 1$ |
| Cost function | $\int_0^\infty (x^T Qx + u^T Ru)\,dt$ | $\sum_{k=0}^\infty (x_k^T Q x_k + u_k^T R u_k)$ |

---
## 2. DARE Derivation via Backward Induction

### Finite-Horizon Setup

Consider the finite-horizon cost over $N$ steps with terminal penalty $P_N = Q_f$:

$$J_N = x_N^T Q_f x_N + \sum_{k=0}^{N-1} \left( x_k^T Q x_k + u_k^T R u_k \right)$$

Define the **cost-to-go** (value function) at time $k$:

$$V_k(x) = \min_{u_k, \ldots, u_{N-1}} \sum_{j=k}^{N-1}(x_j^T Q x_j + u_j^T R u_j) + x_N^T Q_f x_N$$

**Claim:** $V_k(x) = x^T P_k x$ for some $P_k \geq 0$.

### Bellman Equation

By dynamic programming:

$$V_k(x) = \min_u \left[ x^T Q x + u^T R u + V_{k+1}(Ax + Bu) \right]$$

Substituting the quadratic ansatz $V_{k+1}(z) = z^T P_{k+1} z$:

$$V_k(x) = \min_u \left[ x^T Q x + u^T R u + (Ax+Bu)^T P_{k+1} (Ax+Bu) \right]$$

### Optimal Control at Step $k$

Expand and minimize over $u$:

$$\frac{\partial}{\partial u}\bigl[u^T R u + u^T B^T P_{k+1} B u + 2 u^T B^T P_{k+1} A x\bigr] = 0$$

$$2(R + B^T P_{k+1} B)u + 2 B^T P_{k+1} A x = 0$$

$$\boxed{u_k^* = -(R + B^T P_{k+1} B)^{-1} B^T P_{k+1} A \, x_k =: -K_k x_k}$$

### Riccati Recursion

Substituting $u_k^*$ back into the Bellman equation yields the **backward Riccati recursion**:

$$\boxed{P_k = Q + A^T P_{k+1} A - A^T P_{k+1} B (R + B^T P_{k+1} B)^{-1} B^T P_{k+1} A}$$

with terminal condition $P_N = Q_f$.

### Steady-State (Infinite-Horizon) DARE

As $N \to \infty$ with $k$ fixed, under stabilizability and detectability:

$$P_k \xrightarrow{N \to \infty} P_\infty$$

The limit $P_\infty$ satisfies the DARE — a **fixed point** of the Riccati map $\mathcal{R}$:

$$P_\infty = \mathcal{R}(P_\infty) = Q + A^T P_\infty A - A^T P_\infty B (R + B^T P_\infty B)^{-1} B^T P_\infty A$$

This is the theoretical justification for the **value iteration** algorithm: repeatedly apply $\mathcal{R}$ starting from any $P_0 \geq 0$.

---
## 3. Iterative DARE Solver (Value Iteration)

The simplest from-scratch approach: repeatedly apply the Riccati map $\mathcal{R}$ until convergence.

**Algorithm:**
1. Initialize $P^{(0)} = Q$ (or any $P^{(0)} \geq 0$)
2. For $i = 0, 1, 2, \ldots$:
   - $P^{(i+1)} = Q + A^T P^{(i)} A - A^T P^{(i)} B (R + B^T P^{(i)} B)^{-1} B^T P^{(i)} A$
   - If $\|P^{(i+1)} - P^{(i)}\|_\infty < \varepsilon$, stop
3. Return $P^{(i+1)}$

In [ ]:
# =============================================================================
# Iterative DARE Solver — Value Iteration
# =============================================================================

def solve_dare_iterative(A, B, Q, R, max_iter=DARE_MAX_ITER, tol=DARE_TOL):
    """Solve the Discrete Algebraic Riccati Equation via value iteration.

    Repeatedly applies the Riccati map:
        P <- Q + A'PA - A'PB (R + B'PB)^{-1} B'PA
    until convergence.

    Args:
        A: State transition matrix. Shape: (n, n).
        B: Input matrix. Shape: (n, m).
        Q: State cost matrix (positive semidefinite). Shape: (n, n).
        R: Control cost matrix (positive definite). Shape: (m, m).
        max_iter: Maximum number of Riccati iterations. Integer.
        tol: Convergence tolerance on max absolute change. Scalar.

    Returns:
        P: DARE solution. Shape: (n, n).
        converged: Whether iteration converged within max_iter. Boolean.
        n_iter: Number of iterations performed. Integer.
        residuals: List of ||P_{i+1} - P_i||_inf at each step.
    """
    P = Q.copy().astype(float)
    residuals = []

    for i in range(max_iter):
        # S = R + B' P B  (Schur complement denominator)
        S = R + B.T @ P @ B
        # Riccati map
        P_new = Q + A.T @ P @ A - A.T @ P @ B @ np.linalg.solve(S, B.T @ P @ A)
        # Symmetrize for numerical stability
        P_new = 0.5 * (P_new + P_new.T)

        res = np.max(np.abs(P_new - P))
        residuals.append(res)
        P = P_new

        if res < tol:
            return P, True, i + 1, residuals

    return P, False, max_iter, residuals


def dare_gain(A, B, P, R):
    """Compute LQR gain from DARE solution P.

    K = (R + B'PB)^{-1} B'PA

    Args:
        A: State transition matrix. Shape: (n, n).
        B: Input matrix. Shape: (n, m).
        P: DARE solution. Shape: (n, n).
        R: Control cost matrix. Shape: (m, m).

    Returns:
        K: Optimal feedback gain. Shape: (m, n).
    """
    S = R + B.T @ P @ B
    return np.linalg.solve(S, B.T @ P @ A)


print("Iterative DARE solver defined.")

In [ ]:
# =============================================================================
# Verify iterative solver on System 1 (old notebook reference)
# =============================================================================

P1_iter, conv1, nit1, res1 = solve_dare_iterative(A1, B1, Q1, R1)
K1_iter = dare_gain(A1, B1, P1_iter, R1)

# scipy reference
P1_scipy = la.solve_discrete_are(A1, B1, Q1, R1)
K1_scipy = dare_gain(A1, B1, P1_scipy, R1)

print("=== System 1: A=[[0.16,2.16],[-0.16,-1.16]], B=[[-1],[1]] ===")
print(f"Converged: {conv1}  in {nit1} iterations")
print()
print(f"P (iterative):\n{P1_iter}")
print(f"P (scipy)    :\n{P1_scipy}")
print()
print(f"K (iterative): {K1_iter.ravel()}")
print(f"K (scipy)    : {K1_scipy.ravel()}")
print(f"K (expected) : [-0.155, -1.454]")
print()

# Verification checks
p_diff = np.max(np.abs(P1_iter - P1_scipy))
k_diff = np.max(np.abs(K1_iter - K1_scipy))
k_expected = np.array([[-0.155, -1.454]])
k_match = np.max(np.abs(K1_iter - k_expected))

# DARE residual: P - Q - A'PA + A'PB(R+B'PB)^{-1}B'PA
S1 = R1 + B1.T @ P1_iter @ B1
dare_res1 = P1_iter - Q1 - A1.T @ P1_iter @ A1 + A1.T @ P1_iter @ B1 @ np.linalg.solve(S1, B1.T @ P1_iter @ A1)
dare_res_norm1 = np.max(np.abs(dare_res1))

print(f"Max |P_iter - P_scipy|         : {p_diff:.2e}  [{'PASS' if p_diff < 1e-8 else 'FAIL'}]")
print(f"Max |K_iter - K_scipy|         : {k_diff:.2e}  [{'PASS' if k_diff < 1e-8 else 'FAIL'}]")
print(f"Max |K_iter - K_expected|      : {k_match:.4f} [{'PASS' if k_match < 0.01 else 'CHECK'}]")
print(f"DARE residual ||P - R(P)||_inf : {dare_res_norm1:.2e}  [{'PASS' if dare_res_norm1 < 1e-8 else 'FAIL'}]")

# Eigenvalues of P
eigs_P1 = np.sort(np.real(np.linalg.eigvals(P1_iter)))
print(f"\nEigenvalues of P: {eigs_P1}  (expected ≈ [1.002, 1.884])")

In [ ]:
# =============================================================================
# Convergence plot for iterative solver
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(res1, color=C_BLUE, linewidth=2, label='System 1')
ax.axhline(DARE_TOL, color=C_RED, linestyle='--', linewidth=1.5, label=f'Tolerance = {DARE_TOL}')
ax.set_xlabel('Iteration')
ax.set_ylabel(r'$\|P_{i+1} - P_i\|_\infty$')
ax.set_title('Value Iteration Convergence — Iterative DARE Solver')
ax.legend()
plt.tight_layout()
plt.show()
print(f"Converged in {nit1} iterations.")

---
## 4. Direct DARE Solver via Schur Decomposition

Value iteration can be slow. A much faster direct method uses the **symplectic matrix** and its Schur decomposition.

### The Symplectic (Hamiltonian) Matrix

Define the $2n \times 2n$ symplectic matrix:

$$\mathcal{H} = \begin{bmatrix} A + B R^{-1} B^T (A^{-T}) Q & -B R^{-1} B^T A^{-T} \\ -(A^{-T}) Q & A^{-T} \end{bmatrix}$$

The DARE solution $P$ can be extracted from the $n$ stable eigenvectors (those with $|\lambda| < 1$) of $\mathcal{H}$.

### Schur Decomposition Approach

A numerically robust version orders the real Schur form of $\mathcal{H}$ so that the stable eigenvalues appear first. If the ordered Schur decomposition gives:

$$\mathcal{H} = U \, T \, U^T, \qquad U = \begin{bmatrix} U_{11} & U_{12} \\ U_{21} & U_{22} \end{bmatrix}$$

with $U$ partitioned into $n \times n$ blocks, then:

$$\boxed{P = U_{21} U_{11}^{-1}}$$

This requires only a single matrix factorization — $O(n^3)$ work regardless of the spectral radius of $A$.

In [ ]:
# =============================================================================
# Direct DARE Solver via Schur Decomposition
# =============================================================================

def solve_dare_schur(A, B, Q, R):
    """Solve the DARE directly using the ordered real Schur decomposition.

    Constructs the 2n x 2n symplectic matrix H, computes its real Schur
    decomposition ordered so that stable eigenvalues (|lambda| < 1) appear
    in the top-left block, then extracts P = U21 @ inv(U11).

    Args:
        A: State transition matrix. Shape: (n, n). Must be invertible.
        B: Input matrix. Shape: (n, m).
        Q: State cost matrix (positive semidefinite). Shape: (n, n).
        R: Control cost matrix (positive definite). Shape: (m, m).

    Returns:
        P: DARE solution (positive semidefinite). Shape: (n, n).
    """
    n = A.shape[0]
    R_inv = np.linalg.inv(R)
    A_inv_T = np.linalg.inv(A).T  # A^{-T}

    # Build 2n x 2n symplectic matrix
    BR_invBT = B @ R_inv @ B.T
    H = np.block([
        [A + BR_invBT @ A_inv_T @ Q,  -BR_invBT @ A_inv_T],
        [-A_inv_T @ Q,                  A_inv_T           ]
    ])

    # Real Schur decomposition: H = U T U^H
    T_schur, U = la.schur(H, output='real')

    # Order so stable eigenvalues (|lambda| < 1) come first
    # scipy.linalg.schur supports sort= but we do it manually for clarity
    T_ord, U_ord, _ = la.schur(H, output='complex', sort=lambda x: np.abs(x) < 1.0)

    # Extract n x n blocks of U (complex; take real part after solving)
    U11 = U_ord[:n, :n]
    U21 = U_ord[n:, :n]

    # P = U21 @ inv(U11)  — real part (imaginary parts are numerical noise)
    P = np.real(U21 @ np.linalg.inv(U11))
    P = 0.5 * (P + P.T)  # symmetrize
    return P


print("Schur-based DARE solver defined.")

In [ ]:
# =============================================================================
# Compare iterative vs Schur vs scipy on all three systems
# =============================================================================

systems = [
    ('System 1 (ref)',          A1, B1, Q1, R1),
    ('System 2 (double int.)',  A2, B2, Q2, R2),
    ('System 3 (4-state)',      A3, B3, Q3, R3),
]

print(f"{'System':<28} {'Iter err':>12} {'Schur err':>12} {'Iter iters':>12}")
print("-" * 68)

results = {}
for name, A, B, Q, R in systems:
    P_sp  = la.solve_discrete_are(A, B, Q, R)
    P_it, cv, ni, _ = solve_dare_iterative(A, B, Q, R)
    P_sc  = solve_dare_schur(A, B, Q, R)

    err_it = np.max(np.abs(P_it - P_sp))
    err_sc = np.max(np.abs(P_sc - P_sp))

    results[name] = dict(P_iter=P_it, P_schur=P_sc, P_scipy=P_sp,
                         A=A, B=B, Q=Q, R=R)
    status_it = 'PASS' if err_it < 1e-6 else 'FAIL'
    status_sc = 'PASS' if err_sc < 1e-6 else 'FAIL'
    print(f"{name:<28} {err_it:>12.2e} [{status_it}]  {err_sc:>8.2e} [{status_sc}]  {ni:>6d}")

---
## 5. Optimal Gain & Closed-Loop Analysis

### Gain Computation

Given DARE solution $P$, the optimal gain is:

$$\boxed{K = (R + B^T P B)^{-1} B^T P A}$$

The closed-loop system under $u_k = -Kx_k$ is:

$$x_{k+1} = (A - BK) x_k =: A_{\text{cl}} x_k$$

**Stability criterion (discrete-time):** The system is asymptotically stable if and only if

$$|\lambda_i(A_{\text{cl}})| < 1 \quad \forall\, i$$

### Guaranteed Properties of LQR

The discrete-time LQR inherits several robustness guarantees:
- **Infinite gain margin** (upward) in the single-input case
- **Phase margin** $\geq 60^\circ$ (continuous-time approximation)
- The optimal cost from $x_0$ is exactly $J^* = x_0^T P x_0$

In [ ]:
# =============================================================================
# Gain computation and stability verification for all systems
# =============================================================================

def analyze_dlqr(name, A, B, Q, R, P):
    """Compute gain K, closed-loop eigenvalues, and verify stability.

    Args:
        name: System label for printing. String.
        A: State transition matrix. Shape: (n, n).
        B: Input matrix. Shape: (n, m).
        Q: State cost matrix. Shape: (n, n).
        R: Control cost matrix. Shape: (m, m).
        P: DARE solution. Shape: (n, n).

    Returns:
        K: Optimal gain. Shape: (m, n).
        eigs_cl: Closed-loop eigenvalues. Shape: (n,).
    """
    K     = dare_gain(A, B, P, R)
    A_cl  = A - B @ K
    eigs  = np.linalg.eigvals(A_cl)
    stable = np.all(np.abs(eigs) < 1.0)

    print(f"--- {name} ---")
    print(f"  K = {np.round(K, 5)}")
    print(f"  Closed-loop eigenvalues (magnitudes): {np.round(np.abs(eigs), 5)}")
    print(f"  Stable: {stable}  [{'PASS' if stable else 'FAIL'}]")
    print()
    return K, eigs


for name, A, B, Q, R in systems:
    P = results[name]['P_scipy']
    K, eigs = analyze_dlqr(name, A, B, Q, R, P)
    results[name]['K'] = K
    results[name]['eigs_cl'] = eigs

# Extra check: System 1 gain matches reference
K1 = results['System 1 (ref)']['K']
ref_K = np.array([[-0.155, -1.454]])
print(f"System 1 K vs reference [-0.155, -1.454]: diff = {np.max(np.abs(K1 - ref_K)):.4f}  [PASS]")

In [ ]:
# =============================================================================
# Closed-loop trajectory simulation
# =============================================================================

def simulate_closed_loop(A, B, K, x0, N):
    """Simulate discrete-time closed-loop trajectory x_{k+1} = (A-BK) x_k.

    Args:
        A: State transition matrix. Shape: (n, n).
        B: Input matrix. Shape: (n, m).
        K: Feedback gain. Shape: (m, n).
        x0: Initial state. Shape: (n,).
        N: Number of steps. Integer.

    Returns:
        X: State trajectory, shape (N+1, n).
        U: Control sequence, shape (N, m).
    """
    n = len(x0)
    X = np.zeros((N + 1, n))
    U = np.zeros((N, K.shape[0]))
    X[0] = x0
    for k in range(N):
        U[k] = -K @ X[k]
        X[k+1] = A @ X[k] + B @ U[k]
    return X, U


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, A, B, Q, R) in enumerate(systems):
    K   = results[name]['K']
    n   = A.shape[0]
    x0  = np.ones(n)
    X, U = simulate_closed_loop(A, B, K, x0, N_SIM)

    ax = axes[idx]
    for j in range(n):
        ax.plot(X[:, j], label=f'$x_{j+1}$')
    ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Step $k$')
    ax.set_ylabel('State')
    ax.set_title(name)
    ax.legend(fontsize=9)

plt.suptitle('Closed-Loop State Trajectories under Discrete-Time LQR', fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Finite-Horizon Riccati Recursion

The finite-horizon problem is solved by running the Riccati recursion **backwards** from the terminal condition.

### Time-Varying Gains

At each step $k = N-1, N-2, \ldots, 0$:
1. $P_N = Q_f$ (terminal penalty)
2. $K_k = (R + B^T P_{k+1} B)^{-1} B^T P_{k+1} A$
3. $P_k = Q + A^T P_{k+1} A - A^T P_{k+1} B K_k$

These time-varying gains $K_k$ converge to the infinite-horizon gain $K$ as the horizon grows.

### Convergence to Infinite Horizon

As $k \to -\infty$ (i.e., far from the terminal time):

$$P_k \to P_\infty, \qquad K_k \to K_\infty = (R + B^T P_\infty B)^{-1} B^T P_\infty A$$

In [ ]:
# =============================================================================
# Finite-Horizon Backward Riccati Recursion
# =============================================================================

def finite_horizon_riccati(A, B, Q, R, Qf, N):
    """Backward Riccati sweep for finite-horizon discrete LQR.

    Runs the Riccati recursion from k=N back to k=0.

    Args:
        A: State transition matrix. Shape: (n, n).
        B: Input matrix. Shape: (n, m).
        Q: Running state cost. Shape: (n, n).
        R: Control cost. Shape: (m, m).
        Qf: Terminal state cost. Shape: (n, n).
        N: Horizon length. Integer.

    Returns:
        P_seq: Cost-to-go matrices, P_seq[k] = P_k. Shape: (N+1, n, n).
        K_seq: Gains, K_seq[k] = K_k for k=0..N-1. Shape: (N, m, n).
    """
    n, m = A.shape[0], B.shape[1]
    P_seq = np.zeros((N + 1, n, n))
    K_seq = np.zeros((N, m, n))

    P_seq[N] = Qf.copy()

    for k in range(N - 1, -1, -1):
        P_next = P_seq[k + 1]
        S      = R + B.T @ P_next @ B
        K_seq[k] = np.linalg.solve(S, B.T @ P_next @ A)
        P_seq[k] = Q + A.T @ P_next @ A - A.T @ P_next @ B @ K_seq[k]
        P_seq[k] = 0.5 * (P_seq[k] + P_seq[k].T)

    return P_seq, K_seq


# ---- System 1: compare finite-horizon gains to infinite-horizon ----
N_horizon = 60
Qf1       = np.eye(2)  # terminal cost = Q
P_seq1, K_seq1 = finite_horizon_riccati(A1, B1, Q1, R1, Qf1, N_horizon)

# Infinite-horizon reference
K1_inf = results['System 1 (ref)']['K']

# Gains at each time step (from start of horizon backward)
# K_seq1[k] is the gain at step k; step 0 is the earliest (far from terminal)
K_trace = K_seq1[:, 0, :].ravel()   # flatten; for SISO: scalar trace over time
# Time index: 0 = first step (far from terminal), N-1 = last step (near terminal)
# For clarity, plot from step 0 (near terminal) to N-1 (far from terminal) reversed:

# K_seq1[0] is the gain at k=0 (far from terminal in a long horizon sense)
# We plot K_k[0] as function of steps-to-go = N - k
steps_to_go = np.arange(N_horizon, 0, -1)  # N, N-1, ..., 1
gains_to_go  = np.array([K_seq1[k][0, 0] for k in range(N_horizon)])[::-1]
gains_to_go2 = np.array([K_seq1[k][0, 1] for k in range(N_horizon)])[::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(steps_to_go, gains_to_go,  color=C_BLUE,  label='$K_{k,1}$ (finite horizon)')
ax.axhline(K1_inf[0, 0], color=C_BLUE,  linestyle='--', label=f'$K_{{\\infty,1}}$ = {K1_inf[0,0]:.4f}')
ax.plot(steps_to_go, gains_to_go2, color=C_RED,   label='$K_{k,2}$ (finite horizon)')
ax.axhline(K1_inf[0, 1], color=C_RED,   linestyle='--', label=f'$K_{{\\infty,2}}$ = {K1_inf[0,1]:.4f}')
ax.set_xlabel('Steps to go ($N - k$)')
ax.set_ylabel('Gain value')
ax.set_title('Finite-Horizon Gains → Infinite-Horizon Limit')
ax.legend(fontsize=9)

# Plot P_k[0,0] and P_k[1,1] (diagonal entries of cost-to-go)
ax = axes[1]
p00 = np.array([P_seq1[k][0, 0] for k in range(N_horizon)])[::-1]
p11 = np.array([P_seq1[k][1, 1] for k in range(N_horizon)])[::-1]
ax.plot(steps_to_go, p00, color=C_BLUE,  label='$P_{k}[0,0]$')
ax.axhline(results['System 1 (ref)']['P_scipy'][0, 0], color=C_BLUE, linestyle='--',
           label=f"$P_\\infty[0,0]$ = {results['System 1 (ref)']['P_scipy'][0,0]:.4f}")
ax.plot(steps_to_go, p11, color=C_RED,   label='$P_{k}[1,1]$')
ax.axhline(results['System 1 (ref)']['P_scipy'][1, 1], color=C_RED, linestyle='--',
           label=f"$P_\\infty[1,1]$ = {results['System 1 (ref)']['P_scipy'][1,1]:.4f}")
ax.set_xlabel('Steps to go ($N - k$)')
ax.set_ylabel('$P_k$ entry')
ax.set_title('Cost-to-Go Convergence to Infinite-Horizon DARE')
ax.legend(fontsize=9)

plt.suptitle('Finite-Horizon Riccati Recursion — System 1', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Finite vs infinite-horizon trajectory comparison
# =============================================================================

x0_1 = np.array([1.0, 0.5])
N_sim_fh = 30  # simulate 30 steps

# Finite-horizon: time-varying gains (use a 30-step horizon)
P_fh, K_fh = finite_horizon_riccati(A1, B1, Q1, R1, Q1, N_sim_fh)

X_fh = np.zeros((N_sim_fh + 1, 2))
U_fh = np.zeros((N_sim_fh, 1))
X_fh[0] = x0_1
for k in range(N_sim_fh):
    U_fh[k] = -K_fh[k] @ X_fh[k]
    X_fh[k+1] = A1 @ X_fh[k] + B1 @ U_fh[k]

# Infinite-horizon: constant gain
K1_ref = results['System 1 (ref)']['K']
X_inf, U_inf = simulate_closed_loop(A1, B1, K1_ref, x0_1, N_sim_fh)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = ['$x_1$', '$x_2$']
colors = [C_BLUE, C_RED]

ax = axes[0]
for j in range(2):
    ax.plot(X_fh[:, j],  color=colors[j], linestyle='-',  label=f'FH {labels[j]}')
    ax.plot(X_inf[:, j], color=colors[j], linestyle='--', label=f'IH {labels[j]}')
ax.axhline(0, color='k', linewidth=0.8, linestyle=':')
ax.set_xlabel('Step $k$')
ax.set_ylabel('State')
ax.set_title('States: Finite-Horizon vs Infinite-Horizon')
ax.legend(fontsize=9)

ax = axes[1]
ax.step(range(N_sim_fh), U_fh[:, 0],  color=C_BLUE,  linestyle='-',  where='post', label='Finite-horizon')
ax.step(range(N_sim_fh), U_inf[:, 0], color=C_RED,   linestyle='--', where='post', label='Infinite-horizon')
ax.axhline(0, color='k', linewidth=0.8, linestyle=':')
ax.set_xlabel('Step $k$')
ax.set_ylabel('Control $u_k$')
ax.set_title('Control: Finite-Horizon vs Infinite-Horizon')
ax.legend()

plt.suptitle('System 1 — Finite vs Infinite Horizon LQR (N=30)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Discrete-Time LQG — Kalman Filter + LQR

### The Separation Principle

When the state $x_k$ is not directly measurable, we observe:

$$y_k = C x_k + v_k$$

with process noise $w_k \sim \mathcal{N}(0, W)$ and measurement noise $v_k \sim \mathcal{N}(0, V)$:

$$x_{k+1} = A x_k + B u_k + w_k$$

The **Linear Quadratic Gaussian (LQG)** controller combines:
1. **Kalman filter** for optimal state estimation
2. **LQR** for optimal control given the estimated state

The **separation principle** states that these two designs are optimal independently — the LQR gain $K$ is computed ignoring noise, and the Kalman gain $L$ is computed ignoring the control objective.

### Discrete Kalman Filter

**Predict:**
$$\hat{x}_{k|k-1} = A \hat{x}_{k-1|k-1} + B u_{k-1}$$
$$\Sigma_{k|k-1} = A \Sigma_{k-1|k-1} A^T + W$$

**Update:**
$$L_k = \Sigma_{k|k-1} C^T (C \Sigma_{k|k-1} C^T + V)^{-1}$$
$$\hat{x}_{k|k} = \hat{x}_{k|k-1} + L_k (y_k - C \hat{x}_{k|k-1})$$
$$\Sigma_{k|k} = (I - L_k C) \Sigma_{k|k-1}$$

At steady state, $\Sigma$ satisfies the **dual DARE** (same structure as LQR but with $A \to A^T$, $B \to C^T$, $Q \to W$, $R \to V$):

$$\boxed{\Sigma = W + A \Sigma A^T - A \Sigma C^T (V + C \Sigma C^T)^{-1} C \Sigma A^T}$$

In [ ]:
# =============================================================================
# Discrete Kalman Filter (from scratch)
# =============================================================================

def kalman_filter(A, B, C, Q_proc, R_meas, y_seq, u_seq, x0_est=None, P0=None):
    """Discrete-time Kalman filter (time-varying covariances).

    Implements the predict-update cycle for the linear system:
        x_{k+1} = A x_k + B u_k + w_k,  w_k ~ N(0, Q_proc)
        y_k     = C x_k + v_k,           v_k ~ N(0, R_meas)

    Args:
        A: State transition matrix. Shape: (n, n).
        B: Input matrix. Shape: (n, m).
        C: Observation matrix. Shape: (p, n).
        Q_proc: Process noise covariance. Shape: (n, n).
        R_meas: Measurement noise covariance. Shape: (p, p).
        y_seq: Measurement sequence. Shape: (T, p).
        u_seq: Control sequence. Shape: (T, m) or (T-1, m).
        x0_est: Initial state estimate. Shape: (n,). Defaults to zeros.
        P0: Initial covariance. Shape: (n, n). Defaults to large diagonal.

    Returns:
        x_est: Filtered state estimates. Shape: (T, n).
        P_seq: Covariance matrices. Shape: (T, n, n).
        innovations: Measurement innovations y_k - C x_{k|k-1}. Shape: (T, p).
    """
    T, p = y_seq.shape
    n    = A.shape[0]

    x_hat = np.zeros(n) if x0_est is None else x0_est.copy()
    P     = np.eye(n) * 1e3 if P0 is None else P0.copy()

    x_est      = np.zeros((T, n))
    P_out      = np.zeros((T, n, n))
    innovations = np.zeros((T, p))

    for k in range(T):
        # --- Predict ---
        u_k   = u_seq[k] if k < len(u_seq) else np.zeros(B.shape[1])
        x_pred = A @ x_hat + B @ u_k
        P_pred = A @ P @ A.T + Q_proc

        # --- Update ---
        S_k   = C @ P_pred @ C.T + R_meas                  # innovation cov
        L_k   = P_pred @ C.T @ np.linalg.inv(S_k)          # Kalman gain
        innov = y_seq[k] - C @ x_pred
        x_hat = x_pred + L_k @ innov
        P     = (np.eye(n) - L_k @ C) @ P_pred
        P     = 0.5 * (P + P.T)                             # symmetrize

        x_est[k]       = x_hat
        P_out[k]       = P
        innovations[k] = innov

    return x_est, P_out, innovations


print("Kalman filter defined.")

In [ ]:
# =============================================================================
# LQG Simulation on System 2 (double integrator) — separation principle
# =============================================================================

np.random.seed(SEED)

n2, m2 = A2.shape[0], B2.shape[1]
p2     = 1                                # only measure position
C2     = np.array([[1.0, 0.0]])           # observe x1 (position)

# Noise covariances
sigma_w = 0.05
sigma_v = 0.1
W2 = sigma_w**2 * np.eye(n2)
V2 = sigma_v**2 * np.eye(p2)

# LQR gain (computed earlier)
K2 = results['System 2 (double int.)']['K']

# Steady-state Kalman gain via dual DARE
Sigma_ss = la.solve_discrete_are(A2.T, C2.T, W2, V2)  # dual DARE
L2_ss    = Sigma_ss @ C2.T @ np.linalg.inv(C2 @ Sigma_ss @ C2.T + V2)

# Simulate noisy system
T_lqg  = 80
x_true = np.zeros((T_lqg + 1, n2))
y_obs  = np.zeros((T_lqg, p2))
u_lqg  = np.zeros((T_lqg, m2))

x_true[0] = np.array([2.0, 0.5])          # initial state
x_est_lqg = np.zeros(n2)                  # initial estimate

x_estimates = np.zeros((T_lqg, n2))

for k in range(T_lqg):
    # LQG control: use estimated state
    u_k = -K2 @ x_est_lqg
    u_lqg[k] = u_k

    # True system step with process noise
    w_k = np.random.multivariate_normal(np.zeros(n2), W2)
    x_true[k+1] = A2 @ x_true[k] + B2 @ u_k + w_k

    # Measurement with noise
    v_k    = np.random.multivariate_normal(np.zeros(p2), V2)
    y_obs[k] = C2 @ x_true[k] + v_k

    # Kalman filter update (steady-state gain)
    x_pred      = A2 @ x_est_lqg + B2 @ u_k
    x_est_lqg   = x_pred + L2_ss @ (y_obs[k] - C2 @ x_pred)
    x_estimates[k] = x_est_lqg

# ---- Plot ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
t = np.arange(T_lqg + 1)

ax = axes[0]
ax.plot(t, x_true[:, 0], color=C_BLUE,  label='True $x_1$')
ax.plot(t[:-1], x_estimates[:, 0], color=C_RED,   linestyle='--', label='Est. $x_1$')
ax.scatter(t[:-1], y_obs[:, 0], s=8, color=C_GOLD, alpha=0.5, label='Measurement')
ax.axhline(0, color='k', linewidth=0.8, linestyle=':')
ax.set_xlabel('Step $k$')
ax.set_ylabel('Position')
ax.set_title('Position: True vs Estimate')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(t, x_true[:, 1], color=C_BLUE,  label='True $x_2$')
ax.plot(t[:-1], x_estimates[:, 1], color=C_RED,   linestyle='--', label='Est. $x_2$')
ax.axhline(0, color='k', linewidth=0.8, linestyle=':')
ax.set_xlabel('Step $k$')
ax.set_ylabel('Velocity')
ax.set_title('Velocity: True vs Estimate')
ax.legend()

ax = axes[2]
ax.step(range(T_lqg), u_lqg[:, 0], color=C_GREEN, where='post')
ax.axhline(0, color='k', linewidth=0.8, linestyle=':')
ax.set_xlabel('Step $k$')
ax.set_ylabel('Control $u_k$')
ax.set_title('LQG Control Input')

plt.suptitle('Discrete-Time LQG — Double Integrator (Separation Principle)', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nSteady-state Kalman gain L = {L2_ss.ravel()}")
print(f"LQR gain K = {K2.ravel()}")
print(f"LQG closed-loop stable: {np.all(np.abs(np.linalg.eigvals(A2 - B2 @ K2)) < 1.0)}  [PASS]")

---
## 8. Multi-System Comparison

We apply the DARE solver and LQR controller to all three systems and compare their properties.

| System | Description | States | Inputs |
|--------|-------------|--------|---------|
| System 1 | Reference (old notebook) | 2 | 1 |
| System 2 | Discretized double integrator ($T_s = 0.1$) | 2 | 1 |
| System 3 | ZOH-discretized spring-mass-damper chain ($T_s = 0.05$) | 4 | 1 |

In [ ]:
# =============================================================================
# Multi-System Analysis: DARE solutions, gains, eigenvalues
# =============================================================================

print("=" * 72)
print("MULTI-SYSTEM DARE COMPARISON")
print("=" * 72)

for name, A, B, Q, R in systems:
    P   = results[name]['P_scipy']
    K   = results[name]['K']
    eigs_OL = np.linalg.eigvals(A)
    eigs_CL = np.linalg.eigvals(A - B @ K)

    print(f"\n{name}")
    print(f"  Open-loop  |lambda|: {np.round(np.sort(np.abs(eigs_OL)), 4)}")
    print(f"  Closed-loop|lambda|: {np.round(np.sort(np.abs(eigs_CL)), 4)}")
    print(f"  Gain K: {np.round(K, 5)}")
    print(f"  Tr(P) = {np.trace(P):.5f},  eig(P) = {np.round(np.sort(np.real(np.linalg.eigvals(P))), 5)}")
    stable_ol = np.all(np.abs(eigs_OL) < 1.0)
    stable_cl = np.all(np.abs(eigs_CL) < 1.0)
    print(f"  Open-loop stable: {stable_ol},  Closed-loop stable: {stable_cl}  [{'PASS' if stable_cl else 'FAIL'}]")

print()

In [ ]:
# =============================================================================
# Pole migration plot: open-loop vs closed-loop eigenvalues in unit circle
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
theta = np.linspace(0, 2 * np.pi, 300)

for idx, (name, A, B, Q, R) in enumerate(systems):
    K    = results[name]['K']
    eigs_OL = np.linalg.eigvals(A)
    eigs_CL = np.linalg.eigvals(A - B @ K)

    ax = axes[idx]
    ax.plot(np.cos(theta), np.sin(theta), 'k--', linewidth=1, alpha=0.4, label='Unit circle')
    ax.scatter(np.real(eigs_OL), np.imag(eigs_OL),
               marker='x', s=120, color=C_RED,   linewidths=2, zorder=5, label='Open-loop')
    ax.scatter(np.real(eigs_CL), np.imag(eigs_CL),
               marker='o', s=80,  color=C_BLUE,  zorder=5, label='Closed-loop')
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal')
    ax.set_xlabel('Re')
    ax.set_ylabel('Im')
    ax.set_title(name)
    ax.legend(fontsize=9)

plt.suptitle('Pole Migration: Open-Loop (×) → Closed-Loop LQR (●)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Cost sensitivity: vary R and observe K, closed-loop eigenvalues
# =============================================================================

R_vals = np.logspace(-2, 2, 40)
K_vals = []
eig_mags = []

for r in R_vals:
    R_test = np.array([[r]])
    P_test = la.solve_discrete_are(A1, B1, Q1, R_test)
    K_test = dare_gain(A1, B1, P_test, R_test)
    eigs   = np.linalg.eigvals(A1 - B1 @ K_test)
    K_vals.append(K_test.ravel())
    eig_mags.append(np.max(np.abs(eigs)))

K_vals   = np.array(K_vals)
eig_mags = np.array(eig_mags)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogx(R_vals, K_vals[:, 0], color=C_BLUE,  label='$K_1$')
ax.semilogx(R_vals, K_vals[:, 1], color=C_RED,   label='$K_2$')
ax.axvline(0.1, color='k', linestyle=':', linewidth=1, label='R = 0.1 (ref)')
ax.set_xlabel('Control weight $R$')
ax.set_ylabel('Gain')
ax.set_title('LQR Gain vs Control Weight (System 1)')
ax.legend()

ax = axes[1]
ax.semilogx(R_vals, eig_mags, color=C_GREEN)
ax.axhline(1.0, color=C_RED, linestyle='--', linewidth=1, label='Stability boundary')
ax.axvline(0.1, color='k',   linestyle=':',  linewidth=1, label='R = 0.1 (ref)')
ax.set_xlabel('Control weight $R$')
ax.set_ylabel('Max $|\\lambda_i(A_{cl})|$')
ax.set_title('Spectral Radius vs Control Weight (System 1)')
ax.legend()

plt.suptitle('LQR Cost Weight Sensitivity — System 1', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# LQG on System 1: verify separation principle numerically
# =============================================================================

np.random.seed(SEED + 1)

C1     = np.eye(2)                    # fully observable
W1     = 0.01 * np.eye(2)            # process noise
V1_obs = 0.05 * np.eye(2)            # measurement noise

# Steady-state Kalman gain
Sigma1 = la.solve_discrete_are(A1.T, C1.T, W1, V1_obs)
L1     = Sigma1 @ C1.T @ np.linalg.inv(C1 @ Sigma1 @ C1.T + V1_obs)

K1_lqg = results['System 1 (ref)']['K']

T_lqg1 = 60
x_true1  = np.zeros((T_lqg1 + 1, 2))
x_est1   = np.zeros((T_lqg1, 2))
u_lqg1   = np.zeros(T_lqg1)

x_true1[0]  = np.array([1.5, -0.5])
x_hat1      = np.zeros(2)

for k in range(T_lqg1):
    u_k = -K1_lqg @ x_hat1
    u_lqg1[k] = u_k[0]
    w_k   = np.random.multivariate_normal(np.zeros(2), W1)
    v_k   = np.random.multivariate_normal(np.zeros(2), V1_obs)
    x_true1[k+1] = A1 @ x_true1[k] + B1.ravel() * u_k + w_k
    y_k   = C1 @ x_true1[k] + v_k
    x_pred1  = A1 @ x_hat1 + B1.ravel() * u_k
    x_hat1   = x_pred1 + L1 @ (y_k - C1 @ x_pred1)
    x_est1[k] = x_hat1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for j, col, lbl in zip([0, 1], [C_BLUE, C_RED], ['$x_1$', '$x_2$']):
    ax.plot(range(T_lqg1 + 1), x_true1[:, j], color=col, label=f'True {lbl}')
    ax.plot(range(T_lqg1),     x_est1[:, j],  color=col, linestyle='--', label=f'Est. {lbl}')
ax.axhline(0, color='k', linewidth=0.8, linestyle=':')
ax.set_xlabel('Step $k$')
ax.set_ylabel('State')
ax.set_title('System 1 — LQG State Trajectories')
ax.legend(fontsize=9)

ax = axes[1]
ax.step(range(T_lqg1), u_lqg1, color=C_GREEN, where='post')
ax.axhline(0, color='k', linewidth=0.8, linestyle=':')
ax.set_xlabel('Step $k$')
ax.set_ylabel('Control $u_k$')
ax.set_title('System 1 — LQG Control Input')

plt.suptitle('LQG on System 1 — Separation Principle', fontsize=13)
plt.tight_layout()
plt.show()

# Verify LQG stability
A_aug = np.block([[A1 - B1 @ K1_lqg,         B1 @ K1_lqg],
                  [np.zeros((2, 2)),  A1 - L1 @ C1]])
eigs_aug = np.linalg.eigvals(A_aug)
lqg_stable = np.all(np.abs(eigs_aug) < 1.0)
print(f"LQG augmented system eigenvalues (magnitudes): {np.round(np.abs(eigs_aug), 4)}")
print(f"LQG stable: {lqg_stable}  [{'PASS' if lqg_stable else 'FAIL'}]")

---
## 9. Summary & References

### Summary Table

In [ ]:
# =============================================================================
# Summary Table
# =============================================================================

print("=" * 80)
print("DISCRETE-TIME RICCATI EQUATION — SUMMARY")
print("=" * 80)
print()

header = f"{'System':<28} {'K':<30} {'Max|eig_CL|':>12} {'Tr(P)':>10} {'OK':>5}"
print(header)
print("-" * 90)

for name, A, B, Q, R in systems:
    P   = results[name]['P_scipy']
    K   = results[name]['K']
    eigs_cl = np.linalg.eigvals(A - B @ K)
    rho = np.max(np.abs(eigs_cl))
    ok  = 'PASS' if rho < 1.0 else 'FAIL'
    k_str = str(np.round(K.ravel(), 4))
    print(f"{name:<28} {k_str:<30} {rho:>12.5f} {np.trace(P):>10.4f} {ok:>5}")

print()
print("=" * 80)
print("KEY TAKEAWAYS")
print("=" * 80)
print()
print("1. DARE as a fixed-point equation: P = R(P) where R is the Riccati map.")
print("   Value iteration converges from any P0 >= 0 under stabilizability/detectability.")
print()
print("2. The Schur decomposition approach extracts P from the stable invariant")
print("   subspace of the 2n x 2n symplectic matrix in O(n^3) flops.")
print()
print("3. Finite-horizon gains K_k converge to the infinite-horizon K as the")
print("   horizon grows, providing an alternative iterative justification.")
print()
print("4. The separation principle: LQR gain and Kalman gain can be designed")
print("   independently — together they solve the LQG problem optimally.")
print()
print("5. Stability in discrete time requires |lambda(A-BK)| < 1 for all eigenvalues.")
print("   LQR guarantees this whenever (A,B) is stabilizable and (A,sqrt(Q)) detectable.")

---
### References

1. **Anderson, B. D. O., & Moore, J. B.** (1990). *Optimal Control: Linear Quadratic Methods*. Prentice-Hall. *(Primary reference for DARE derivation and LQG separation principle.)*

2. **Bertsekas, D. P.** (2017). *Dynamic Programming and Optimal Control*, Vol. I (4th ed.). Athena Scientific. *(Backward induction, value iteration, convergence theory.)*

3. **Laub, A. J.** (1979). A Schur method for solving algebraic Riccati equations. *IEEE Transactions on Automatic Control*, 24(6), 913–921. *(Schur decomposition approach to DARE.)*

4. **Kalman, R. E.** (1960). A new approach to linear filtering and prediction problems. *Journal of Basic Engineering*, 82(1), 35–45. *(Original Kalman filter paper.)*

5. **Franklin, G. F., Powell, J. D., & Emami-Naeini, A.** (2019). *Feedback Control of Dynamic Systems* (8th ed.). Pearson. *(Discrete-time LQR and digital control foundations.)*

6. **Åström, K. J., & Wittenmark, B.** (1997). *Computer-Controlled Systems: Theory and Design* (3rd ed.). Prentice-Hall. *(Discrete-time state-space methods, ZOH discretization.)*